In [1]:
import os
import re
import time
import argparse
import hashlib
from typing import List, Dict, Tuple

from pypdf import PdfReader
from tqdm import tqdm

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from opensearchpy import OpenSearch
from opensearchpy.helpers import bulk
os.environ["OPENAI_API_KEY"]=""


# -----------------------
# CONFIG (edit as needed)
# -----------------------
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
MIN_CHUNK_CHARS = 50

BULK_BATCH_SIZE = 200  # docs per bulk request

INDEX_NAME = "rag_pdf_index"
VECTOR_FIELD = "embedding"
TEXT_FIELD = "text"

# ---- OpenSearch connection ----
OS_HOST = os.getenv("OPENSEARCH_HOST", "10.103.6.142")
OS_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
OS_USER = os.getenv("OPENSEARCH_USER", "admin")
OS_PASS = os.getenv("OPENSEARCH_PASS", "Aa1Bb2Cc3Dd4")  # set env var instead of hardcoding

# HTTPS / SSL handling
USE_SSL = True
VERIFY_CERTS = False          # quick test; set True + CA_CERTS_PATH for production
CA_CERTS_PATH = None          # e.g. "/path/to/root-ca.pem" if VERIFY_CERTS=True
SSL_SHOW_WARN = False

# Lucene HNSW tuning (OpenSearch 3.x)
HNSW_M = 24
HNSW_EF_CONSTRUCTION = 256


# -----------------------
# Cleaning utilities
# -----------------------
def normalize_unicode(text: str) -> str:
    import unicodedata
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\x00", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def fix_pdf_hyphenation(text: str) -> str:
    # "exam-\nple" -> "example"
    text = re.sub(r"(\w)-\s+(\w)", r"\1\2", text)
    # join broken newlines inside sentences
    text = re.sub(r"([^\n])\n([^\n])", r"\1 \2", text)
    return text


def remove_repeated_headers_footers(
    pages: List[str],
    sample_pages: int = 4,
    top_n: int = 2,
    bottom_n: int = 2
) -> List[str]:
    """
    Heuristic: find repeated top/bottom lines across early pages and remove them.
    """
    def top_lines(p: str) -> List[str]:
        lines = [l.strip() for l in p.splitlines() if l.strip()]
        return lines[:top_n]

    def bottom_lines(p: str) -> List[str]:
        lines = [l.strip() for l in p.splitlines() if l.strip()]
        return lines[-bottom_n:] if len(lines) >= bottom_n else lines

    header_counts: Dict[str, int] = {}
    footer_counts: Dict[str, int] = {}

    for p in pages[:sample_pages]:
        for h in top_lines(p):
            if len(h) <= 120:
                header_counts[h] = header_counts.get(h, 0) + 1
        for f in bottom_lines(p):
            if len(f) <= 120:
                footer_counts[f] = footer_counts.get(f, 0) + 1

    repeated_headers = {k for k, v in header_counts.items() if v >= 2}
    repeated_footers = {k for k, v in footer_counts.items() if v >= 2}

    cleaned = []
    for p in pages:
        lines = p.splitlines()
        while lines and lines[0].strip() in repeated_headers:
            lines.pop(0)
        while lines and lines[-1].strip() in repeated_footers:
            lines.pop(-1)
        cleaned.append("\n".join(lines))
    return cleaned


def is_low_value_chunk(text: str) -> bool:
    if len(text.strip()) < MIN_CHUNK_CHARS:
        return True
    if re.search(r"\b(click here|read more|subscribe|cookie|privacy policy)\b", text, flags=re.I):
        return True
    return False


def sha_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


# -----------------------
# PDF extraction
# -----------------------
def extract_pdf_pages(path: str) -> Tuple[List[str], Dict]:
    reader = PdfReader(path)
    pages = []
    for p in reader.pages:
        try:
            pages.append(p.extract_text() or "")
        except Exception:
            pages.append("")
    meta = {
        "filename": os.path.basename(path),
        "num_pages": len(pages),
        "title": getattr(getattr(reader, "metadata", None), "title", None) or os.path.basename(path),
    }
    return pages, meta


# -----------------------
# Chunking
# -----------------------
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

def make_chunks_from_pdf(path: str) -> List[Dict]:
    pages, meta = extract_pdf_pages(path)

    pages = [normalize_unicode(fix_pdf_hyphenation(p)) for p in pages]
    pages = remove_repeated_headers_footers(pages)

    joined = "\n\n".join([f"[PAGE {i+1}]\n{p}" for i, p in enumerate(pages) if p.strip()])
    joined = normalize_unicode(joined)

    raw_chunks = splitter.split_text(joined)

    out = []
    for idx, chunk in enumerate(raw_chunks):
        if is_low_value_chunk(chunk):
            continue
        out.append({
            "text": chunk.strip(),
            "metadata": {
                "source": meta["filename"],
                "title": meta["title"],
                "num_pages": meta["num_pages"],
                "chunk_index": idx,
                "ingested_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }
        })
    return out


def exact_dedupe(chunks: List[Dict]) -> List[Dict]:
    seen = set()
    out = []
    for c in chunks:
        h = sha_hash(c["text"])
        if h in seen:
            continue
        seen.add(h)
        out.append(c)
    return out


# -----------------------
# OpenSearch index creation (Lucene engine, OpenSearch 3.x safe)
# -----------------------
def connect_opensearch() -> OpenSearch:
    kwargs = dict(
        hosts=[{"host": OS_HOST, "port": OS_PORT}],
        http_auth=(OS_USER, OS_PASS),
        use_ssl=USE_SSL,
        verify_certs=VERIFY_CERTS,
        ssl_show_warn=SSL_SHOW_WARN,
    )
    if USE_SSL and VERIFY_CERTS and CA_CERTS_PATH:
        kwargs["ca_certs"] = CA_CERTS_PATH
    return OpenSearch(**kwargs)


def create_index_if_needed(client: OpenSearch, index_name: str, dimension: int):
    if client.indices.exists(index=index_name):
        return

    # IMPORTANT:
    # - Use engine="lucene" (nmslib is deprecated/blocked for new indices in OpenSearch 3.x)
    # - Put m/ef_construction under method.parameters
    body = {
        "settings": {
            "index": {
                "knn": True
            }
        },
        "mappings": {
            "properties": {
                TEXT_FIELD: {"type": "text"},
                "source": {"type": "keyword"},
                "title": {"type": "keyword"},
                "num_pages": {"type": "integer"},
                "chunk_index": {"type": "integer"},
                "ingested_at": {"type": "date"},
                VECTOR_FIELD: {
                    "type": "knn_vector",
                    "dimension": dimension,
                    "method": {
                        "name": "hnsw",
                        "engine": "lucene",
                        "space_type": "cosinesimil",
                        "parameters": {
                            "m": HNSW_M,
                            "ef_construction": HNSW_EF_CONSTRUCTION
                        }
                    }
                }
            }
        }
    }

    client.indices.create(index=index_name, body=body)
    print(f"[OpenSearch] Created index: {index_name} (engine=lucene)")


# -----------------------
# Bulk upsert
# -----------------------
def bulk_index(client: OpenSearch, index_name: str, items: List[Dict], embeddings: OpenAIEmbeddings) -> int:
    texts = [x["text"] for x in items]
    vectors = embeddings.embed_documents(texts)

    actions = []
    for item, vec in zip(items, vectors):
        md = item["metadata"]
        doc = {
            TEXT_FIELD: item["text"],
            VECTOR_FIELD: vec,
            "source": md.get("source"),
            "title": md.get("title"),
            "num_pages": md.get("num_pages"),
            "chunk_index": md.get("chunk_index"),
            "ingested_at": md.get("ingested_at"),
            "metadata": md,
        }
        actions.append({"_index": index_name, "_source": doc})

    success, errors = bulk(client, actions, raise_on_error=False)
    if errors:
        print("[OpenSearch] Bulk errors (first 3):", errors[:3])
    return int(success)


# -----------------------
# Main
# -----------------------
def ingest_folder(folder: str):
    pdf_paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(".pdf"):
                pdf_paths.append(os.path.join(root, f))

    if not pdf_paths:
        print(f"No PDFs found under: {folder}")
        return

    print(f"Found {len(pdf_paths)} PDF(s). Extracting + cleaning + chunking...")
    chunks: List[Dict] = []
    for p in tqdm(pdf_paths, desc="PDFs", unit="file"):
        try:
            chunks.extend(make_chunks_from_pdf(p))
        except Exception as e:
            print(f"[WARN] Failed {p}: {e}")

    print(f"Generated {len(chunks)} chunks. Running exact dedupe...")
    chunks = exact_dedupe(chunks)
    print(f"{len(chunks)} chunks remain after dedupe.")

    embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)
    dim = len(embeddings.embed_documents(["dimension_probe"])[0])

    client = connect_opensearch()
    info = client.info()
    print("[OpenSearch] Connected. Cluster:", info.get("cluster_name", "unknown"))

    create_index_if_needed(client, INDEX_NAME, dim)

    total = 0
    for i in tqdm(range(0, len(chunks), BULK_BATCH_SIZE), desc="Bulk ingest", unit="batch"):
        batch = chunks[i:i + BULK_BATCH_SIZE]
        total += bulk_index(client, INDEX_NAME, batch, embeddings)

    print(f"Done. Indexed ~{total} docs into {INDEX_NAME}.")


# if __name__ == "__main__":
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--folder", required=True, help="Folder path containing PDFs (recursive)")
#     args = parser.parse_args()
#     ingest_folder(args.folder)

In [2]:
ingest_folder(r"C:\Users\surya.adatravu\Documents\ComplexRAG\pdfs")

Found 1 PDF(s). Extracting + cleaning + chunking...


PDFs: 100%|████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.67file/s]


Generated 70 chunks. Running exact dedupe...
70 chunks remain after dedupe.
[OpenSearch] Connected. Cluster: opensearch
[OpenSearch] Created index: rag_pdf_index (engine=lucene)


Bulk ingest: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:05<00:00,  5.23s/batch]

Done. Indexed ~70 docs into rag_pdf_index.
